# 05 — Phase 2 Attack Analysis
Compares RobustFL performance across attacker ratios (10%, 30%, 50%). Requires Phase 2 experiment results CSVs.

In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parent))
from src.configs.paths import RESULTS_DIR, ARTIFACTS_DIR

# Load results for each attacker ratio
scenarios = {
    '0% (Clean)':  'round_results.csv',
    '10% Attack':  'round_results_robust_ratio_10pct.csv',
    '30% Attack':  'round_results_robust_ratio_30pct.csv',
    '50% Attack':  'round_results_robust_ratio_50pct.csv',
}

dfs = {}
for label, fname in scenarios.items():
    path = RESULTS_DIR / fname
    if path.exists():
        dfs[label] = pd.read_csv(path)
        print(f'Loaded {label}: {len(dfs[label])} rounds')
    else:
        print(f'MISSING: {fname} — run Phase 2 experiments first')

In [ ]:
# Macro F1 vs Round for each attacker ratio
colors = ['steelblue', 'orange', 'red', 'darkred']
fig, ax = plt.subplots(figsize=(12, 5))
for (label, df), color in zip(dfs.items(), colors):
    ax.plot(df['round'], df['macro_f1'], marker='o', markersize=3, label=label, color=color)
ax.axvline(10.5, color='gray', linestyle='--', alpha=0.5, label='Attack start (Round 11)')
ax.set_xlabel('FL Round')
ax.set_ylabel('Macro F1-Score')
ax.set_title('RobustFL — Macro F1 vs Attacker Ratio')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'plots' / 'phase2_macro_f1_ratios.png', dpi=150)
plt.show()

In [ ]:
# FPR vs Round for each attacker ratio
fig, ax = plt.subplots(figsize=(12, 5))
for (label, df), color in zip(dfs.items(), colors):
    ax.plot(df['round'], df['fpr'], marker='s', markersize=3, label=label, color=color)
ax.axvline(10.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('FL Round')
ax.set_ylabel('False Positive Rate')
ax.set_title('RobustFL — FPR vs Attacker Ratio')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'plots' / 'phase2_fpr_ratios.png', dpi=150)
plt.show()

In [ ]:
# Trust score heatmap for the 30% attack scenario
trust_path = RESULTS_DIR / 'trust_scores_robust_ratio_30pct.csv'
if trust_path.exists():
    trust_df = pd.read_csv(trust_path)
    pivot = trust_df.pivot(index='client_id', columns='round', values='trust_score')
    fig, ax = plt.subplots(figsize=(16, 10))
    sns.heatmap(pivot, cmap='RdYlGn', center=0, linewidths=0.2, ax=ax)
    ax.set_title('Trust Scores — 30% Attacker Ratio (attackers should show low scores after Round 11)')
    ax.axvline(10.5, color='white', linewidth=2)
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / 'plots' / 'phase2_trust_heatmap_30pct.png', dpi=150)
    plt.show()
else:
    print('Run Phase 2 (30% attackers) experiment first.')

In [ ]:
# Summary table: final round metrics per scenario
rows = []
for label, df in dfs.items():
    last = df.iloc[-1]
    rows.append({'Scenario': label, 'Final Macro F1': round(last['macro_f1'], 4),
                 'Final Accuracy': round(last['accuracy'], 4), 'Final FPR': round(last['fpr'], 4)})
if rows:
    print(pd.DataFrame(rows).set_index('Scenario').to_string())